# Train **atlas** on Google Colab (free GPU)

One-click unsupervised contrastive training for ImageNette, then the few-label evaluation.

**First:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
A T4 (16 GB) fits `--batch 512 @ 160px` easily and finishes ~300 epochs in ~30–60 min.
Nothing to install or configure — Colab ships CUDA PyTorch, and atlas turns on mixed precision automatically.

In [ ]:
# 1) confirm the GPU
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime→GPU')

In [ ]:
# 2) clone the repo + install the light deps (torch is already on Colab)
!git clone -q https://github.com/cleoanka/atlas.git
%cd atlas
!pip install -q numpy scipy scikit-learn matplotlib pillow

In [ ]:
# 3) fetch ImageNette (~95 MB)
!mkdir -p data && curl -sL https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz -o data/imagenette2-160.tgz
!tar -xzf data/imagenette2-160.tgz -C data/ && echo 'ready:' && ls data/imagenette2-160

In [ ]:
# 4) TRAIN (unsupervised). T4 16GB fits this comfortably; AMP is auto-on on CUDA.
#    Stronger run (slower): add  --backbone resnet50 --epochs 800
!python src/train_contrastive_imagenette.py --epochs 300 --img 160 --batch 512

In [ ]:
# 5) evaluate + figures with the freshly trained encoder
!python src/evaluate.py --dataset imagenette
!python src/sweep.py    --dataset imagenette
!python figures/make_imagenette_figures.py

In [ ]:
# 6) download the trained artifacts before the session ends (Colab is ephemeral)
from google.colab import files
for f in ['data/imagenette_emb.npz', 'data/imagenette_train_emb.npz',
          'models/imagenette_encoder.pt', 'results/imagenette_results.json']:
    try: files.download(f)
    except Exception as e: print('skip', f, e)

### Keep it (optional)

- **Google Drive** instead of downloading: `from google.colab import drive; drive.mount('/content/drive')` then `!cp data/imagenette_emb.npz models/imagenette_encoder.pt /content/drive/MyDrive/`.
- **Push back to the repo:** set a token, `!git config user.email you@x` / `user.name you`, then commit the new `data/imagenette_emb.npz` + `results/*.json` + `figures/imagenette_*.png` and `git push`.
- The `.pt` encoder is what you keep to embed new images later; the `*_emb.npz` files are what the repo's evaluation/figures use.